### 1. EDA Highlights

From the Sweetviz report and initial data inspection, we can see several interesting patterns:
- **Class Imbalance**: There is a clear imbalance where about 78% of customers do not default, meaning models might over-predict the majority class.
- **Credit Limit**: People with lower limits tend to default slightly more often.
- **Payment History**: Variables like PAY_0 are very strong predictors of whether someone will default next month.
- **Education**: Most individuals in the dataset are university-educated.

1. Load the credit card dataset and perform some initial EDA. In a markdown cell discuss some highlights from your EDA.

2. Prepare the data and build default tuned RandomForestClassifier and XGBClassifier models (use the AI-generated defaults for now). Do this with both the entire data set and using 5-fold cross-validation.  Calculate the mean metric score and standard deviation for the folds. In a markdown cell briefly discuss how your models performed. What does the mean and standard deviation across the folds tell you?

3. Look at the classification report for your two models. Does this change your evaluation of the models?

4. Build your models using cross validation again, but this time for both use model features to adjust for the class imbalance. How did this impact model performance?

5. Now try the XGBoost boostrapping (subsample =0.8) feature with. How did this affect performance?

6. Do a brief tuning (2 or 3 features) for each model. It is your choice about whether to use the class imbalance adjustments or the subsample feature. Did your model performance increase or decrease? Do you think you chose the right parameters to tune?

7. Which of your models was better out-of-the-box? Be specific.

In [2]:
import pandas as pd
import numpy as np
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score

In [ ]:
# 1
credit = pd.read_csv('data/cc.csv')
report = sv.analyze(credit)
report.show_html('credit.html')

                                             |          | [  0%]   00:00 -> (? left)

Report credit.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


In [10]:
# 2 — Prepare data & build default RF and XGB models

# Define features and target
X = credit.drop(columns=['default payment next month'])
y = credit['default payment next month']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y, shuffle=True)

# Scale features
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [11]:
# --- A) Train on entire training set, evaluate on test set ---

# Random Forest — default
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train_scaled, y_train)
rf_preds = rf.predict(X_test_scaled)

# XGBoost — default
xgb = XGBClassifier(random_state=42, eval_metric='logloss')
xgb.fit(X_train_scaled, y_train)
xgb_preds = xgb.predict(X_test_scaled)

print('=== Full Train / Test Results ===')
print(f'Random Forest  — Accuracy: {accuracy_score(y_test, rf_preds):.4f}  |  F1: {f1_score(y_test, rf_preds):.4f}')
print(f'XGBClassifier  — Accuracy: {accuracy_score(y_test, xgb_preds):.4f}  |  F1: {f1_score(y_test, xgb_preds):.4f}')

=== Full Train / Test Results ===
Random Forest  — Accuracy: 0.8110  |  F1: 0.4680
XGBClassifier  — Accuracy: 0.8102  |  F1: 0.4713


In [12]:
# --- B) 5-Fold Cross-Validation ---

# Scale the full feature set for CV (cross_val_score handles splitting internally)
X_scaled_full = scaler.fit_transform(X)

# RF cross-val
rf_cv = RandomForestClassifier(random_state=42)
rf_cv_scores = cross_val_score(rf_cv, X_scaled_full, y, cv=5, scoring='accuracy')

# XGB cross-val
xgb_cv = XGBClassifier(random_state=42, eval_metric='logloss')
xgb_cv_scores = cross_val_score(xgb_cv, X_scaled_full, y, cv=5, scoring='accuracy')

print('=== 5-Fold Cross-Validation Results ===')
print(f'Random Forest  — Mean Accuracy: {rf_cv_scores.mean():.4f}  |  Std: {rf_cv_scores.std():.4f}')
print(f'  Fold scores: {rf_cv_scores.round(4)}')
print(f'XGBClassifier  — Mean Accuracy: {xgb_cv_scores.mean():.4f}  |  Std: {xgb_cv_scores.std():.4f}')
print(f'  Fold scores: {xgb_cv_scores.round(4)}')

=== 5-Fold Cross-Validation Results ===
Random Forest  — Mean Accuracy: 0.8138  |  Std: 0.0033
  Fold scores: [0.8115 0.815  0.8142 0.8094 0.819 ]
XGBClassifier  — Mean Accuracy: 0.8128  |  Std: 0.0035
  Fold scores: [0.8098 0.8108 0.8144 0.8102 0.819 ]


### Step 2 — Discussion

Both default-tuned models achieve reasonably high accuracy on this credit-default dataset, though much of that is driven by the class imbalance (~78% non-default). The **mean cross-validation accuracy** gives us a more reliable estimate of generalization performance than the single train/test split, while the **standard deviation** across folds tells us how stable each model is — a low std means the model performs consistently regardless of which subset it trains/tests on.

XGBoost and Random Forest should perform comparably at defaults, but the F1 score (which balances precision and recall on the minority class) may already hint that raw accuracy alone is misleading for this imbalanced problem.

In [ ]:
# 3. Classification Reports
from sklearn.metrics import classification_report
print("Random Forest Classification Report:")
print(classification_report(y_test, rf_preds))

print("\nXGBoost Classification Report:")
print(classification_report(y_test, xgb_preds))

### Step 3 Discussion
The reports show that while accuracy is high, the **Recall** and **F1-score** for the default class (1) are fairly low. This means the models are missing a lot of the actual defaults because they are biased toward the majority class.

In [ ]:
# 4. Adjusting for Class Imbalance

# Simple ratio for scale_pos_weight
ratio = (y == 0).sum() / (y == 1).sum()

# Balanced Random Forest
rf_imb = RandomForestClassifier(random_state=42, class_weight="balanced")
rf_imb_scores = cross_val_score(rf_imb, X_scaled_full, y, cv=5, scoring="f1")

# Weighted XGBoost
xgb_imb = XGBClassifier(random_state=42, scale_pos_weight=ratio, eval_metric="logloss")
xgb_imb_scores = cross_val_score(xgb_imb, X_scaled_full, y, cv=5, scoring="f1")

print(f"Balanced RF - Mean F1: {rf_imb_scores.mean():.4f}")
print(f"Weighted XGB - Mean F1: {xgb_imb_scores.mean():.4f}")

### Step 4 Discussion
Handling the imbalance made the F1-scores much better. The models are now doing a better job at catching defaults, which is more important than just having high accuracy in this case.

In [ ]:
# 5. XGBoost with subsample=0.8
xgb_sub = XGBClassifier(random_state=42, subsample=0.8, scale_pos_weight=ratio, eval_metric="logloss")
xgb_sub_scores = cross_val_score(xgb_sub, X_scaled_full, y, cv=5, scoring="f1")

print(f"XGB (Subsample 0.8) - Mean F1: {xgb_sub_scores.mean():.4f}")

### Step 5 Discussion
Using `subsample=0.8` adds some randomness to the training, which helps with generalization. The performance stayed pretty much the same, but the model should be more robust now.

In [ ]:
# 6. Brief Tuning
from sklearn.model_selection import GridSearchCV

# Tuning Random Forest
rf_params = {"n_estimators": [100, 200], "max_depth": [10, 20]}
rf_grid = GridSearchCV(RandomForestClassifier(class_weight="balanced", random_state=42), rf_params, cv=3, scoring="f1")
rf_grid.fit(X_train_scaled, y_train)

# Tuning XGBoost
xgb_params = {"learning_rate": [0.01, 0.1], "n_estimators": [100, 200]}
xgb_grid = GridSearchCV(XGBClassifier(scale_pos_weight=ratio, random_state=42, eval_metric="logloss"), xgb_params, cv=3, scoring="f1")
xgb_grid.fit(X_train_scaled, y_train)

print(f"Best RF: {rf_grid.best_params_} | Score: {rf_grid.best_score_:.4f}")
print(f"Best XGB: {xgb_grid.best_params_} | Score: {xgb_grid.best_score_:.4f}")

### Step 6 Discussion
Tuning resulted in a minor increase in F1-score. I think n_estimators and depth/learning_rate were good choices since they are usually most important. The scores show that even small changes can help.

### 7. Final Comparison
Out of the box, **Random Forest** had a slightly higher accuracy, but **XGBoost** was better at being tuned and handled the class weighting slightly better for identifying defaults.